# Clase 083 — Voting classifiers: hard y soft

Combinamos varios modelos heterogéneos en un ensemble por votación y comparamos
**hard voting** (voto por mayoría) contra **soft voting** (promedio de probabilidades),
apoyándonos en el principio de *wisdom of the crowd*.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Dataset `make_moons` y split 80/20

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

np.random.seed(42)

X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('train', X_train.shape, 'test', X_test.shape)

## 2. Tres modelos base heterogéneos por separado

In [ ]:
log_clf = LogisticRegression(random_state=42)
rf_clf  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
svc_clf = SVC(probability=True, random_state=42)

acc_base = {}
for name, clf in [('LogReg', log_clf), ('RandomForest', rf_clf), ('SVC', svc_clf)]:
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    acc_base[name] = acc
    print(f'{name:14s} acc test: {acc:.4f}')

best_base = max(acc_base.values())
print(f'\nmejor modelo base: {best_base:.4f}')

## 3. Hard voting (voto por mayoría)

In [ ]:
hard = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rf_clf), ('svc', svc_clf)],
    voting='hard')
hard.fit(X_train, y_train)
acc_hard = accuracy_score(y_test, hard.predict(X_test))
print(f'Hard voting acc test: {acc_hard:.4f}')

## 4. Soft voting (promedio de probabilidades)

Requiere que todos los modelos expongan `predict_proba` (por eso `SVC(probability=True)`).

In [ ]:
soft = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rf_clf), ('svc', svc_clf)],
    voting='soft')
soft.fit(X_train, y_train)
acc_soft = accuracy_score(y_test, soft.predict(X_test))
print(f'Soft voting acc test: {acc_soft:.4f}')

# El ensemble por votación no debería quedar por debajo del promedio de sus bases.
assert acc_soft >= np.mean(list(acc_base.values())), 'soft voting no aporta'
print('assert OK: soft voting >= promedio de los modelos base')

## 5. Ponderar con `weights` favoreciendo al Random Forest

In [ ]:
weighted = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rf_clf), ('svc', svc_clf)],
    voting='soft', weights=[1, 2, 1])
weighted.fit(X_train, y_train)
acc_w = accuracy_score(y_test, weighted.predict(X_test))
print(f'Soft voting weights=[1,2,1]: {acc_w:.4f}')

## 6. Comparativa visual

In [ ]:
labels = list(acc_base) + ['Hard', 'Soft', 'Soft w=[1,2,1]']
vals   = list(acc_base.values()) + [acc_hard, acc_soft, acc_w]
colors = ['#aaa', '#aaa', '#aaa', '#37a', '#3a7', '#a73']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, vals, color=colors)
ax.axhline(best_base, ls='--', color='k', lw=0.8, label=f'mejor base = {best_base:.3f}')
ax.set_ylim(min(vals) - 0.03, 1.0)
ax.set_ylabel('accuracy test')
ax.set_title('Voting: hard vs soft vs modelos base')
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Ejercicios

1. Reemplazá un modelo base por una segunda `LogisticRegression` casi idéntica
   (dos modelos correlacionados). Observá cómo el ensemble deja de ganar por falta
   de diversidad.
2. Sacá `probability=True` del `SVC` y comprobá que `voting="soft"` falla con
   `AttributeError` (no hay `predict_proba`).
3. Probá otros `weights` y validalos con `cross_val_score` en vez de elegirlos a ojo.
4. Cambiá el dataset a `load_breast_cancer()` y repetí la comparación hard vs soft.

## Conclusiones

- El ensemble por votación aprovecha la *wisdom of the crowd*: errores independientes
  se cancelan al combinarse.
- Soft voting suele ganarle a hard **si** los modelos están bien calibrados; con
  probabilidades ruidosas, hard puede ser preferible.
- La diversidad (algoritmos distintos) es la condición clave: modelos correlacionados
  no aportan.
- `weights` puede ayudar, pero debe justificarse con validación, no a ojo.